In [1]:
import os
import json
import pandas as pd
import re
import string
import unicodedata
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# Download necessary resources
nltk.download('stopwords')
nltk.download('wordnet')

# Set seed for langdetect to ensure consistent results
DetectorFactory.seed = 0


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\advai\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\advai\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
# Telugu Stopwords List
telugu_stopwords = {
    "అందరూ", "అంత", "అయితే", "అప్పుడు", "అసలు", "ఆ", "ఇది", "ఈ", "ఒక", "కానీ",
    "కూడా", "లో", "పై", "తర్వాత", "వద్ద", "మాత్రమే", "తో", "కు", "కి", "లోపల", "బయట",
    "ఎందుకు", "ఎప్పుడు", "ఏ", "ఏవి", "ఏది", "ఎమి", "ఎక్కడ", "ఎమిటి", "ఎందుకంటే",
    "అయినా", "అయినప్పటికీ", "ఇప్పుడే", "ఇలా", "ఇక", "తర్వాత", "కూడా", "దయచేసి", "చుట్టూ",
    "అటు", "ఇటు", "ఒకటి", "రెండు", "మూడు", "మొత్తం", "సరే", "చూద్దాం", "వద్ద", "లేదు",
    "వాడు", "ఆమె", "అతను", "వాళ్ళు", "మేము", "మీరు", "నన్ను", "నిన్ను", "దాని", "వాటిని",
    "ఎవరైనా", "ఎక్కడైనా", "ఎప్పుడైనా", "ఇదీ", "అలా", "ఇలా", "ఇంకా", "అపైన", "దిగువ",
    "వెనుక", "పక్కన", "మధ్య", "తప్పనిసరి", "మళ్ళీ", "అంతా", "ఎక్కువగా", "తక్కువగా",
    "ఎవరికీ", "ఏవీ", "ఏమైనా", "ఎప్పుడైనా"
}

# Combine Telugu and English stopwords
english_stopwords = set(stopwords.words('english'))
stopwords_combined = telugu_stopwords.union(english_stopwords)

# Initialize NLP Tools
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
normalizer_factory = IndicNormalizerFactory()
telugu_normalizer = normalizer_factory.get_normalizer("te")


In [5]:
def preprocess_text(text):
    """
    Preprocess the text by cleaning, normalizing, and tokenizing.
    """
    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Regex patterns for cleaning
    patterns = [
        r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',  # Email
        r'(\+?\d{1,4}[\s-])?(?:\(\d{1,3}\)[\s-]?)?\d{1,4}[\s-]?\d{1,4}[\s-]?\d{1,9}',  # Phone numbers
        r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b',  # Dates
        r'https?://[^\s/$.?#].[^\s]*',  # URLs
        r'\[(\d+)\]',  # Bracketed numbers
        r'10.\d{4,9}/[-._;()/:A-Z0-9]+',  # DOIs
        r'(?:ISBN(?:-13)?:?\s*)?(?=[-0-9]{13}$|(?=(?:[-0-9]{17}$)|(?:[-0-9X]{10}$))(?:97[89][-0-9]{10}$))\d{1,5}[-\s]?\d{1,7}[-\s]?\d{1,7}[-\s]?\d{1,7}[-\s]?(?:\d|X)',  # ISBNs
        r'[-+]?\d*\.?\d+([eE][-+]?\d+)?',  # Numbers
        r'#\w+',  # Hashtags
        r'@\w+',  # Mentions
        r'\b(?:\d{1,2} [A-Za-z]{3,9} \d{4}|\d{4}/\d{2}/\d{2})\b',  # Dates (expanded formats)
        r'\b(?:\d{1,3}\.){3}\d{1,3}\b',  # IP addresses
        r'<.*?>',  # HTML tags
        r'\((.*?)\)',  # Bracketed content
        r'\b(?:\$|€|₹|£)\d+(?:\.\d{1,2})?\b',  # Currency values
        r'[^\x00-\x7F]+',  # Non-ASCII characters
        r'(.)\1{2,}',  # Repeated characters
        r'(?:[A-Za-z]:)?(?:\\[A-Za-z0-9_.-]+)+\\?',  # File paths
        r'\b0[xX][0-9a-fA-F]+\b',  # Hexadecimal numbers
        r'["\'](.*?)["\']',  # Quoted strings
    ]

    # Apply regex cleaning
    for pattern in patterns:
        text = re.sub(pattern, '', text)

    # Tokenize words
    words = text.split()

    # Normalize and process words based on language
    final_words = []
    for word in words:
        try:
            lang = detect(word)
            if lang == 'te':
                word = telugu_normalizer.normalize(word)
            elif lang == 'en':
                word = stemmer.stem(word)
                word = lemmatizer.lemmatize(word)
            if word.lower() not in stopwords_combined:
                final_words.append(word)
        except LangDetectException:
            # If language detection fails, keep the word as is
            final_words.append(word)

    # Remove empty spaces and tokenize
    final_text = " ".join([word for word in final_words if word.strip()])
    final_text = final_text.translate(str.maketrans("", "", string.punctuation))
    tokens = final_text.split()

    return tokens


In [ ]:
# Define the folder path
folder_path = r"C:\Projects\CodeMix\data\Raw"
all_data = []

# Iterate over each file in the folder
for file in os.listdir(folder_path):
    if file.endswith(".json"):
        file_path = os.path.join(folder_path, file)

        # Open and read the JSON file
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            all_data.extend(data)

# Convert the combined data into a DataFrame
df = pd.DataFrame(all_data)

# Apply preprocessing to the 'text' column
df['cleaned_text'] = df['text'].apply(lambda x: " ".join(preprocess_text(x)))

# Select only the 'cleaned_text' column
cleaned_text_df = df[['cleaned_text']]

# Display the first few rows of the DataFrame
print(cleaned_text_df.head())
